In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier


In [2]:
class AdaBoostStumps:

    def __init__(self, n_stumps=3, random_state=None):
        if not isinstance(n_stumps, int) or n_stumps < 1:
            raise ValueError("n_stumps must be a positive integer.")
        self.n_stumps = n_stumps
        self.random_state = random_state
        self.stumps_ = []
        self.alphas_ = []
        self.classes_ = None
        self.errors_ = []

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel()
        if X.ndim != 2:
            raise ValueError("X must be a 2D feature matrix.")
        if len(X) != len(y) or len(y) == 0:
            raise ValueError("X and y must contain the same non-zero number of rows.")

        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError("AdaBoostStumps supports binary classification (exactly 2 classes).")
        y_signed = np.where(y == self.classes_[1], 1, -1)
        sample_weights = np.full(len(y), 1 / len(y), dtype=float)
        self.stumps_ = []
        self.alphas_ = []
        self.errors_ = []

        for round_index in range(self.n_stumps):
            stump = DecisionTreeClassifier(
                max_depth=1,
                random_state=None if self.random_state is None else self.random_state + round_index,
            )
            stump.fit(X, y_signed, sample_weight=sample_weights)
            pred = stump.predict(X)
            incorrect = pred != y_signed
            error = float(np.dot(sample_weights, incorrect))
            self.errors_.append(error)

            # A stump at chance or worse cannot improve the ensemble.
            if error >= 0.5:
                if not self.stumps_:
                    raise ValueError("No useful stump found: weighted error is at least 0.5.")
                self.errors_.pop()
                break

            # Perfect stump: it already classifies all training examples.
            if error <= 0.0:
                alpha = 1.0
            else:
                error = np.clip(error, 1e-12, 1 - 1e-12)
                alpha = 0.5 * np.log((1 - error) / error)

            self.stumps_.append(stump)
            self.alphas_.append(float(alpha))
            if error <= 0.0:
                break

            sample_weights *= np.exp(-alpha * y_signed * pred)
            sample_weights /= sample_weights.sum()

        return self

    def decision_function(self, X):
        if not self.stumps_:
            raise ValueError("Call fit before making predictions.")
        X = np.asarray(X)
        scores = np.zeros(len(X), dtype=float)
        for alpha, stump in zip(self.alphas_, self.stumps_):
            scores += alpha * stump.predict(X)
        return scores

    def predict(self, X):
        signed = np.where(self.decision_function(X) >= 0, 1, -1)
        return np.where(signed == 1, self.classes_[1], self.classes_[0])

    def stump_summary(self):
        return pd.DataFrame({
            "stump": np.arange(1, len(self.stumps_) + 1),
            "weighted_error": self.errors_,
            "alpha": self.alphas_,
        })


In [11]:
df = pd.DataFrame({
    "X1": [1, 9, 3, 4, 5, 6, 6, 7, 9, 9,23],
    "X2": [5, 3, 4, 8, 1, 9, 5, 8, 9, 2,19],
    "label": [1, 1, 1, 1, 0, 1, 0, 0, 0, 0,1],
})

X = df[["X1", "X2"]]
y = df["label"]

model = AdaBoostStumps(n_stumps=3, random_state=0)
model.fit(X, y)

print(f"Trained {len(model.stumps_)} stumps")
model.stump_summary()


Trained 3 stumps


,stump,weighted_error,alpha
0,1,0.272727,0.490415
1,2,0.187500,0.733169
2,3,0.307692,0.405465


In [12]:
new_points = pd.DataFrame({"X1": [1, 9], "X2": [5, 9]})
predictions = model.predict(new_points)
predictions


array([1, 0])